In [16]:
# --- IMPORT STATEMENTS ---

import numpy as np
import pandas as pd
import re
from astropy.coordinates import SkyCoord
import astropy.units as u

In [17]:
# --- GLOBAL VARIABLES ---

LINUX_DIR = "/home/aimee/mphys"
DATA_DIR = f'{LINUX_DIR}/data'
FILE_DIR = f'{DATA_DIR}/Paladini2003_synthetic.tsv'

In [18]:
# --- FUNCTION DEFINITIONS ---

In [19]:
def get_Paladini_data():

    # get raw data  
    with open(FILE_DIR, 'r') as file: # this file is so ugly istg
        data = []
        for i, line in enumerate(file):
            if i > 37: # skip headers
                arr_raw = re.split(r'[ |]+', line)
                arr = []
                for j in arr_raw:
                    if (j != '') & (j != '|') & (j != '**') & (j != '<'): # this is worse than DA Green catalogue omg
                        arr.append(j)
                # cut array past index 17 (don't need comments, but do need consistent dimensions)
                arr = arr[0:17]
                if '\n' not in arr:
                    data.append(arr)

    # Extract useful info
    ra_arr, dec_arr, ang_diameter_arr, ang_diam_err_arr = [], [], [], [] # add more if needed
    for row in data:
        # RA
        ra_deg = 360/24 * (float(row[5]) + float(row[6])/60 + float(row[7])/3600) # HOURS, minutes, seconds --> degrees
        ra_arr.append(ra_deg) # deg
    
        # Dec
        if float(row[8]) < 0:
            multiplier = -1
        else:
            multiplier = +1
        dec_deg = multiplier * (np.abs(float(row[8])) + float(row[9])/60 + float(row[10])/3600) # degrees, minutes, seconds --> degrees
        dec_arr.append(float(dec_deg)) # deg
    
        # Angular diameter and error
        ang_diameter_arr.append(float(row[13])) # arcmin
        ang_diam_err_arr.append(float(row[14])) # arcmin


    # Convert ra and dec to galactic coords
    l_arr, b_arr = [], []
    for ra, dec, in zip(ra_arr, dec_arr):
        eq_coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')
        galactic_coord = eq_coord.galactic
        l_arr.append(float(galactic_coord.l.value))
        b_arr.append(float(galactic_coord.b.value))
    
    l_cut, b_cut, ang_diameter_cut, ang_diam_err_cut = [], [], [], []
    for l, b, ang_diameter, ang_diam_err in zip(l_arr, b_arr, ang_diameter_arr, ang_diam_err_arr):
        if ang_diameter > 5: # only keep sources > 5 arcmin
            l_cut.append(l)
            b_cut.append(b)
            ang_diameter_cut.append(ang_diameter)
            ang_diam_err_cut.append(ang_diam_err)

    return l_cut, b_cut, ang_diameter_cut, ang_diam_err_cut


In [20]:
l_arr, b_arr, ang_diameter_arr, ang_diam_err_arr = get_Paladini_data()

print(l_arr) # deg
print(b_arr) # deg
print(ang_diameter_arr) # arcmin
print(ang_diam_err_arr) # arcmin

[0.09846448273803338, 0.19873426293437904, 0.19869162583233854, 0.39865912430946376, 2.3987435734525935, 2.8987130349518857, 3.2986979846634283, 4.99867197301628, 5.298556399020071, 5.898599464055603, 5.998715293594503, 5.998705037910113, 6.098671875190018, 6.398570647742769, 6.498686371165514, 6.598681690413033, 6.598677544439852, 6.6986629342832265, 6.8984742695713095, 6.9986675357661605, 6.998700965249696, 7.298617515688242, 7.398693576831179, 7.398530080959899, 7.39861080291036, 8.498638103204456, 8.498682838829781, 9.698729560088601, 10.29853354151907, 10.598506150261903, 10.898831495033475, 11.198611529621635, 11.19856375952209, 11.398517499851053, 11.698650622113654, 12.798789050686814, 12.79867769407929, 13.198687647448235, 13.398696766422917, 13.398703712613504, 13.798568758204928, 14.198834937408654, 14.198765655317727, 14.298572916759133, 14.598713330104196, 14.5986874712475, 14.698774093189575, 14.998732385878355, 15.098655589261625, 15.098696534832126, 16.298744017406243, 